In [0]:
%python
from pyspark.sql import functions as F

BRONZE_PATH = "/mnt/usvikformula1dl/bronze"
transaction_bronze_path = f"{BRONZE_PATH}/transaction_fact"
SILVER_PATH = "/mnt/usvikformula1dl/silver/"

new_transaction_df=spark.read.format("delta").load(F"{SILVER_PATH}/transaction_fact").limit(2)
new_transaction_df_1=new_transaction_df.withColumn("amount",F.lit(5000)).withColumn("status",F.lit("PENDING"))
display(new_transaction_df_1)



                                               

Add a new row to the new dataframe created above 

In [0]:
%python
display()

In [0]:
%python
from pyspark.sql import Row, functions as F
from pyspark.sql import SparkSession
from datetime import datetime

data=[Row(transaction_id=1021, customer_id=110, account_id=5023, transaction_ts=datetime.now(), transaction_type="Debit", amount=5000, currency="USD", status="PENDING")]
df=spark.createDataFrame(data)
new_transaction_df_2=new_transaction_df_1.unionByName(df)
display(new_transaction_df_2)


MERGER with existing transaction dataframe 

In [0]:
%python
from delta.tables import *
from pyspark.sql import functions as F
deltaTable=DeltaTable.forPath(spark,F"{SILVER_PATH}/transaction_fact")

deltaTable.alias('tgt') \
    .merge(
        new_transaction_df_2.alias("src"),
           'tgt.transaction_id=src.transaction_id' \
    ) \
    .whenMatchedUpdateAll() \
    .whenNotMatchedInsertAll() \
    .execute()
       



In [0]:
%python
 spark.read.format("delta").load(F"{SILVER_PATH}/transaction_fact").display()
 

In [0]:
%python
spark.read.format("delta").load("/mnt/usvikformula1dl/silver/transaction_fact").filter("transaction_id=1021").display()